### Import the libraries

In [38]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score, classification_report)
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

In [39]:
np.random.seed(42)
n = 1000

In [40]:
df = pd.DataFrame({
    'study_hours'   : np.random.normal(6, 2, n).clip(0, 12),
    'sleep_hours'   : np.random.normal(7, 1.5, n).clip(3, 10),
    'attendance_pct': np.random.normal(75, 15, n).clip(0, 100),
    'prev_score'    : np.random.normal(65, 15, n).clip(0, 100),
    'stress_level'  : np.random.randint(1, 10, n).astype(float),
    'exercise_days' : np.random.randint(0, 7, n).astype(float)})

In [41]:
score = (
    df['study_hours']    * 4.0 +
    df['attendance_pct'] * 0.3 +
    df['prev_score']     * 0.4 +
    df['sleep_hours']    * 1.5 +
    np.random.normal(0, 5, n))

In [42]:
df['result'] = (score > score.median()).astype(int)

In [43]:
df.head()

,study_hours,sleep_hours,attendance_pct,prev_score,stress_level,exercise_days,result
0,6.993428,9.099033,64.872326,36.382887,8.0,5.0,0
1,5.723471,8.386951,72.832220,52.094225,3.0,1.0,0
2,7.295377,7.089446,63.113701,58.795917,8.0,1.0,1
3,9.046060,6.029595,70.380577,93.315315,8.0,0.0,1
4,5.531693,8.047335,46.595780,73.348297,5.0,5.0,1


### Split the data

In [44]:
X = df.drop('result', axis=1).values
y = df['result'].values

In [45]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, stratify = y, random_state = 42)

In [46]:
print(f"Training samples : {X_train.shape[0]}")
print(f"Test samples     : {X_test.shape[0]}")

Training samples : 800
Test samples     : 200


# Define Base Models

In [47]:
# IMPORTANT: Base models must be DIVERSE
# Using 4 models of the same type (e.g. 4 Random Forests) gives almost no benefit — they all make the same mistakes
# Different algorithms = different mistakes = better stacking

base_models = {
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('model',  RandomForestClassifier(
                       n_estimators = 100,
                       max_depth    = 6,
                       random_state = 42,
                       n_jobs       = -1))
    ]),
    # Good at: non-linear patterns, feature interactions
    # Weakness: can overfit on noisy data

    'XGBoost': Pipeline([
        ('scaler', StandardScaler()),
        ('model',  xgb.XGBClassifier(
                       n_estimators = 100,
                       max_depth    = 4,
                       learning_rate= 0.1,
                       subsample    = 0.8,
                       eval_metric  = 'logloss',
                       random_state = 42,
                       n_jobs       = -1))
    ]),
    # Good at: structured tabular data, handles outliers
    # Weakness: needs more tuning

    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model',  LogisticRegression(
                       C        = 1.0,
                       max_iter = 1000,
                       random_state = 42))
    ]),
    # Good at: linear relationships, calibrated probabilities
    # Weakness: misses non-linear patterns

    'SVM': Pipeline([
        ('scaler', StandardScaler()),
        ('model',  SVC(
                       kernel      = 'rbf',
                       C           = 1.0,
                       probability = True,
                       random_state= 42))
    ]),
    # Good at: high-dimensional data, max margin boundary
    # Weakness: slow on large data, needs probability=True
}

In [48]:
base_models

{'Random Forest': Pipeline(steps=[('scaler', StandardScaler()),
                 ('model',
                  RandomForestClassifier(max_depth=6, n_jobs=-1,
                                         random_state=42))]),
 'XGBoost': Pipeline(steps=[('scaler', StandardScaler()),
                 ('model',
                  XGBClassifier(base_score=None, booster=None, callbacks=None,
                                colsample_bylevel=None, colsample_bynode=None,
                                colsample_bytree=None, device=None,
                                early_stopping_rounds=None,
                                enable_categorical=False, eval_metric='logloss',
                                feature_types=None, feature_weights=None,
                                gamma=None, grow_policy=None,
                                importance_type=None,
                                interaction_constraints=None, learning_rate=0.1,
                                max_bin=None, max_cat_thres

In [49]:
print("Base models defined:")
for name in base_models:
    print(f"  → {name}")

Base models defined:
  → Random Forest
  → XGBoost
  → Logistic Regression
  → SVM


# Generate Out-of-Fold Predictions (The Core Step)

In [50]:
n_folds = 5

In [51]:
skf = StratifiedKFold(n_splits = n_folds, shuffle = True, random_state = 42)
skf

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

In [52]:
# We will fill these arrays with OOF predictions
# Each column = one base model's predictions
# Each row    = one training sample's prediction

oof_preds  = np.zeros((len(X_train), len(base_models)))
# Shape: (800, 4)
# 800 training samples × 4 base models

In [53]:
oof_preds

array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       ...,
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]], shape=(800, 4))

In [54]:
test_preds = np.zeros((len(X_test), len(base_models)))
# Shape: (200, 4)
# 200 test samples × 4 base models

test_preds

array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],


In [56]:
print("Generating Out-of-Fold predictions...\n")
print(f"{'Model':<25} {'Fold 1':>8} {'Fold 2':>8} "
      f"{'Fold 3':>8} {'Fold 4':>8} {'Fold 5':>8} {'Mean':>8}")
print("─" * 75)

for model_idx, (model_name, model) in enumerate(base_models.items()):

    fold_aucs = []

    # This array temporarily stores test predictions
    # from each fold — we average them at the end
    # Why average? Because each fold trains a slightly
    # different model — averaging reduces variance
    test_fold_preds = np.zeros((len(X_test), n_folds))

    for fold_num, (train_idx, val_idx) in enumerate(
            skf.split(X_train, y_train)):

        # Split into fold's train and validation
        X_fold_train = X_train[train_idx]
        X_fold_val   = X_train[val_idx]
        y_fold_train = y_train[train_idx]
        y_fold_val   = y_train[val_idx]

        # Train model on this fold's training portion
        # It has NOT seen val_idx samples at all
        model.fit(X_fold_train, y_fold_train)

        # Predict on validation fold
        # This is the KEY: model predicts on data it
        # has NEVER seen → honest prediction
        val_probs = model.predict_proba(X_fold_val)[:, 1]
        oof_preds[val_idx, model_idx] = val_probs

        # Also predict test set using this fold's model
        test_fold_preds[:, fold_num] = \
            model.predict_proba(X_test)[:, 1]

        # Track AUC for this fold
        fold_auc = roc_auc_score(y_fold_val, val_probs)
        fold_aucs.append(fold_auc)

    # Average test predictions across all 5 fold models
    # Each fold trained a slightly different model
    # Averaging them gives more stable test prediction
    test_preds[:, model_idx] = test_fold_preds.mean(axis=1)

    # Print fold-by-fold AUC scores
    fold_str = '  '.join([f'{a:.4f}' for a in fold_aucs])
    print(f"{model_name:<25} {fold_str}  {np.mean(fold_aucs):.4f}")

print("\nOOF predictions complete!")
print(f"Shape of OOF predictions  : {oof_preds.shape}")
print(f"Shape of Test predictions : {test_preds.shape}")

Generating Out-of-Fold predictions...

Model                       Fold 1   Fold 2   Fold 3   Fold 4   Fold 5     Mean
───────────────────────────────────────────────────────────────────────────
Random Forest             0.9519  0.9156  0.9503  0.9356  0.9322  0.9371
XGBoost                   0.9539  0.9209  0.9605  0.9320  0.9387  0.9412
Logistic Regression       0.9664  0.9489  0.9694  0.9436  0.9595  0.9576
SVM                       0.9612  0.9250  0.9620  0.9336  0.9498  0.9463

OOF predictions complete!
Shape of OOF predictions  : (800, 4)
Shape of Test predictions : (200, 4)


# Train Meta Model

In [57]:
# Now we use the OOF predictions as features to train the meta model

# Think of it this way:
# Original problem: predict Pass/Fail from study_hours, attendance...
# Meta problem:     predict Pass/Fail from [RF_prob, XGB_prob, LR_prob, SVM_prob]

# The meta model learns questions like:
# "If RF says 0.9 and XGB says 0.8, what is the true probability?"
# "When RF and LR disagree, who should I trust more?"

meta_model = LogisticRegression(
    C            = 0.1,
    # ↑ small C = strong regularization
    #   meta model should be SIMPLE and regularized
    #   prevents overfitting on OOF predictions
    max_iter     = 1000,
    random_state = 42
)
# Why Logistic Regression as meta model?
# → Simple → less likely to overfit
# → Interpretable → we can see weights
# → Fast → doesn't add much training time
# → The hard work is done by base models already

meta_model.fit(oof_preds, y_train)
# X = OOF predictions [RF_prob, XGB_prob, LR_prob, SVM_prob]
# y = true labels (same y_train we always use)

# See what the meta model learned
print("Meta Model — How much it trusts each base model:")
print("(higher positive = more trusted)\n")
for model_name, weight in zip(base_models.keys(),
                               meta_model.coef_[0]):
    bar    = '█' * int(abs(weight) * 10)
    direction = '+' if weight > 0 else '-'
    print(f"  {model_name:<25} {direction}{abs(weight):.4f}  {bar}")

Meta Model — How much it trusts each base model:
(higher positive = more trusted)

  Random Forest             +0.9535  █████████
  XGBoost                   +1.1113  ███████████
  Logistic Regression       +1.7086  █████████████████
  SVM                       +1.3918  █████████████


# Make Final Predictions


In [58]:
# Now use the test_preds (base model predictions on test set)
# as input to the trained meta model

final_preds = meta_model.predict(test_preds)
final_probs = meta_model.predict_proba(test_preds)[:, 1]

stacking_acc = accuracy_score(y_test, final_preds)
stacking_auc = roc_auc_score(y_test, final_probs)
stacking_f1  = f1_score(y_test, final_preds)

print("STACKING Final Results:")
print(f"  Accuracy : {stacking_acc:.4f}")
print(f"  ROC-AUC  : {stacking_auc:.4f}")
print(f"  F1 Score : {stacking_f1:.4f}")

STACKING Final Results:
  Accuracy : 0.8600
  ROC-AUC  : 0.9472
  F1 Score : 0.8627


# Compare Each Base Model vs Stacking

In [59]:
# Retrain each base model on full training data
# and compare with stacking

results = {}

for model_name, model in base_models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]

    results[model_name] = {
        'Accuracy': accuracy_score(y_test, preds),
        'AUC'     : roc_auc_score(y_test, probs),
        'F1'      : f1_score(y_test, preds)
    }
    print(f"{model_name:<25} "
          f"Acc={results[model_name]['Accuracy']:.4f}  "
          f"AUC={results[model_name]['AUC']:.4f}")

results['STACKING ✅'] = {
    'Accuracy': stacking_acc,
    'AUC'     : stacking_auc,
    'F1'      : stacking_f1
}

print(f"\n{'STACKING':<25} "
      f"Acc={stacking_acc:.4f}  "
      f"AUC={stacking_auc:.4f}")

# Stacking should beat every individual model
# because it combines their strengths

Random Forest             Acc=0.8450  AUC=0.9265
XGBoost                   Acc=0.8550  AUC=0.9358
Logistic Regression       Acc=0.8750  AUC=0.9505
SVM                       Acc=0.8700  AUC=0.9471

STACKING                  Acc=0.8600  AUC=0.9472


# Blending

In [60]:
# Step 1 — Split training data into two parts
# Base models only ever see blend_train
# They predict on holdout which they never trained on
X_blend_train, X_holdout, y_blend_train, y_holdout = train_test_split(
    X_train, y_train,
    test_size    = 0.3,
    stratify     = y_train,
    random_state = 42
)

print(f"Base model training data : {len(X_blend_train)} samples")
print(f"Holdout (meta training)  : {len(X_holdout)} samples")
print(f"Test (final evaluation)  : {len(X_test)} samples\n")

# Step 2 — Storage arrays
# Rows = holdout samples, Columns = base models
holdout_preds = np.zeros((len(X_holdout), len(base_models)))
test_preds_bl = np.zeros((len(X_test),    len(base_models)))

# Step 3 — Train base models and collect predictions
print(f"{'Model':<25} {'Holdout AUC':>12}")
print("─" * 40)

for idx, (model_name, model) in enumerate(base_models.items()):

    # Train base model on blend_train ONLY
    # It never sees X_holdout during training
    model.fit(X_blend_train, y_blend_train)

    # Predict on holdout
    # These predictions are honest because
    # model was trained on a completely different portion
    holdout_preds[:, idx] = \
        model.predict_proba(X_holdout)[:, 1]

    # Predict on test set as well
    test_preds_bl[:, idx] = \
        model.predict_proba(X_test)[:, 1]

    h_auc = roc_auc_score(y_holdout, holdout_preds[:, idx])
    print(f"{model_name:<25} {h_auc:>12.4f}")

# Step 4 — Train meta model on holdout predictions
# The meta model sees:
#   X = [RF_pred, XGB_pred, LR_pred, SVM_pred] for each holdout sample
#   y = true labels for those holdout samples
blend_meta = LogisticRegression(
    C            = 0.1,
    max_iter     = 1000,
    random_state = 42
)
blend_meta.fit(holdout_preds, y_holdout)

# Step 5 — Final prediction on test set
blend_preds = blend_meta.predict(test_preds_bl)
blend_probs = blend_meta.predict_proba(test_preds_bl)[:, 1]

blend_acc = accuracy_score(y_test, blend_preds)
blend_auc = roc_auc_score(y_test, blend_probs)
blend_f1  = f1_score(y_test, blend_preds)

print(f"\nBLENDING Final Results:")
print(f"  Accuracy : {blend_acc:.4f}")
print(f"  ROC-AUC  : {blend_auc:.4f}")
print(f"  F1 Score : {blend_f1:.4f}")

Base model training data : 560 samples
Holdout (meta training)  : 240 samples
Test (final evaluation)  : 200 samples

Model                      Holdout AUC
────────────────────────────────────────
Random Forest                   0.9256
XGBoost                         0.9241
Logistic Regression             0.9517
SVM                             0.9369

BLENDING Final Results:
  Accuracy : 0.8500
  ROC-AUC  : 0.9492
  F1 Score : 0.8529
